In [1]:
## Defining packages
import os
import shutil
import pathlib
from datetime import datetime
from PIL import Image
from PIL.ExifTags import TAGS
from pathlib import Path
import piexif
import re

## Defining Static Variables
datetime_original_tag = 36867
datetime_digitized_tag = 36868
datetime_tag = 306

In [2]:
source_base = "D:/lara/Snapchat/snapchat_downloads_2020_2021_renamed/"
destination_base = "D:/lara/Snapchat with EXIF/"

In [3]:
def filename_timestamp_splitter(filename):
    
    # extract only digits
    digits = re.sub(r"\D", "", filename)

    # must be exactly 14 digits
    if len(digits) < 14:
        return False

    digits = digits[:14]
    return digits

In [4]:
def format_exif_timestamp(datetime_string):
    
    parsed = datetime.strptime(datetime_string, "%Y%m%d%H%M%S")
    exif_format = parsed.strftime("%Y:%m:%d %H:%M:%S")
    
    return exif_format

In [5]:
def contains_real_datetime(filename):
    
    # extract only digits
    digits = re.sub(r"\D", "", filename)

    # must be exactly 14 digits
    if len(digits) < 14:
        return False

    digits = digits[:14]

    try:
        datetime.strptime(digits, "%Y%m%d%H%M%S")
        return True
    except ValueError:
        return False

In [6]:
def has_exif_datetime(image_path):
    
    img = Image.open(image_path)
    exif_data = img._getexif()
    
    original_datetime = exif_data.get(datetime_original_tag)
    digitized_datetime = exif_data.get(datetime_digitized_tag)
    img_datetime = exif_data.get(datetime_tag)

    true_datetime = original_datetime or digitized_datetime or img_datetime
    
    if true_datetime:
        return True
    else:
        return False

# Steps

1. check if filename has valid datetime name (if not = ignore, no way to set exif data)
2. check if there is exif data for main datetime original (if there is = ignore, no reason to add datetime data)
   - but check first that it matches the filename, flag it if it doesnt
3. check if there is exif data for other datetime values (if there is, must do some validation)
   - if there is no exif datetime data at all -> easy = use filename datetime
   - if there is a datetime value -> check that it matches the filename
      - if it does match the filename -> good to set all datetimes to that value
      - if it does not match the filename -> ignore and flag the file as contradictory
4. copy image to destination
6. add all datetime values with the one found in filename (if previous checks have passed)
7. do some validation that the datetime value has changed and maybe some validation that the rest of the exif hasn't been altered? if not overkill?
8. once the file has been moved (check it exists) and there's a datetime original value, delete the original image from the source path

Note: This code only works reliably for JPG/JPEG files.

In [7]:
contradictary_files = {}

folder = Path(source_base)
for file in folder.rglob("*"):
    
    suffix = Path(str(file)).suffix
    if suffix.upper() not in ['.JPG', '.JPEG']:
        continue
    
    # Defining file-specific variables
    file_source_path = str(file)
    filename = Path(file_source_path).name
    img = Image.open(file_source_path)
    exif_data = img._getexif()
    original_datetime = exif_data.get(datetime_original_tag)
    digitized_datetime = exif_data.get(datetime_digitized_tag)
    plain_datetime = exif_data.get(datetime_tag)
    filename_datetime = filename_timestamp_splitter(filename[:17])
    
    print("Reading: ", file_source_path)
    
    # Step 1: Ignore if there is no datetime found in the filename
    if not contains_real_datetime(filename[:17]):
        print("   Step 1: Failed (filename does not contain datetime data)")
        continue

    print("   Step 1: Passed")
    filename_datetime_formatted = format_exif_timestamp(filename_datetime)
    
    
    # Step 2: Ignore if there is already a datetime found in the EXIF data (but with consolidation checks)
    if has_exif_datetime(file_source_path):
        
        print("   Step 2: Failed (EXIF data already contains datetime data)")
        
        # Step 2.5: Flag the file if the filename datetime is different from the EXIF Datetime original
        if original_datetime != filename_datetime_formatted:
            contradictary_files[file_source_path] = original_datetime
            print("   Step 2.5: Failed")

        continue
    
    print("   Step 2: Passed")
    
    # Step 3: Check that other exif datetime values don't differ from the filename datetime
    if digitized_datetime or plain_datetime:
        other_exif_datetime = digitized_datetime or plain_datetime
        if other_exif_datetime != filename_datetime_formatted:
            contradictary_files[file_source_path] = other_exif_datetime
            print("   Step 3: Failed (Contradictory dates found)")
            continue
    
    print("   Step 3: Passed")
            
    # Happy that the filename will be the datetime value for the exif data
    final_exif_datetime = filename_datetime_formatted
    print("   Finalised datetime string = " + final_exif_datetime)
    
    # Step 4: Copy image to the destination
    file_dest_path = destination_base + filename
    try:
        shutil.copyfile(file_source_path, file_dest_path)
    except:
        print("Error copying file: " + file_path)
        print("   Step 4: Failed (Error copying file: " + file_path + ")")
        continue
    
    print("   Step 4: Passed")

    # Step 5: Add datetime values to exif data if they don't already exist
    with Image.open(file_dest_path) as new_img:
        new_exif_data = new_img._getexif()
        new_exif_dict = piexif.load(new_img.info.get("exif", b""))

        new_exif_dict["Exif"][piexif.ExifIFD.DateTimeOriginal]= final_exif_datetime

        try:
            new_exif_dict["Exif"][piexif.ExifIFD.DateTimeDigitized]
        except:
            new_exif_dict["Exif"][piexif.ExifIFD.DateTimeDigitized] = final_exif_datetime

        try:
            new_exif_dict["0th"][piexif.ImageIFD.DateTime]
        except:
            new_exif_dict["0th"][piexif.ImageIFD.DateTime] = final_exif_datetime

        exif_bytes = piexif.dump(new_exif_dict)

        new_img.save(file_dest_path, exif=exif_bytes)
        img.close()
        new_img.close()
    
    print("   Step 5: Passed")
    
    # Step 6: Validation
    if Path(file_dest_path).exists():
        
        with Image.open(file_dest_path) as verify_img:
            new_exif_data = verify_img._getexif()

        if new_exif_data.get(datetime_original_tag):
            os.remove(file_source_path)
        else:
            print("Datetime data not found in EXIF:", file_dest_path)
            continue
    else:
        print("Copy failed, not deleting:", source)
        continue
    


Reading:  D:\lara\Snapchat\snapchat_downloads_2020_2021_renamed\2020-01-01_134653 - Snapchat.jpg
   Step 1: Passed
   Step 2: Passed
   Step 3: Passed
   Finalised datetime string = 2020:01:01 13:46:53
   Step 4: Passed
   Step 5: Passed
Reading:  D:\lara\Snapchat\snapchat_downloads_2020_2021_renamed\2020-01-01_150151 - Snapchat.jpg
   Step 1: Passed
   Step 2: Passed
   Step 3: Passed
   Finalised datetime string = 2020:01:01 15:01:51
   Step 4: Passed
   Step 5: Passed
Reading:  D:\lara\Snapchat\snapchat_downloads_2020_2021_renamed\2020-01-01_205727 - Snapchat.jpg
   Step 1: Passed
   Step 2: Passed
   Step 3: Passed
   Finalised datetime string = 2020:01:01 20:57:27
   Step 4: Passed
   Step 5: Passed
Reading:  D:\lara\Snapchat\snapchat_downloads_2020_2021_renamed\2020-01-01_205737 - Snapchat.jpg
   Step 1: Passed
   Step 2: Passed
   Step 3: Passed
   Finalised datetime string = 2020:01:01 20:57:37
   Step 4: Passed
   Step 5: Passed
Reading:  D:\lara\Snapchat\snapchat_downloads_20

   Step 5: Passed
Reading:  D:\lara\Snapchat\snapchat_downloads_2020_2021_renamed\2020-01-08_202714 - Snapchat.jpg
   Step 1: Passed
   Step 2: Passed
   Step 3: Passed
   Finalised datetime string = 2020:01:08 20:27:14
   Step 4: Passed
   Step 5: Passed
Reading:  D:\lara\Snapchat\snapchat_downloads_2020_2021_renamed\2020-01-08_204903 - Snapchat.jpg
   Step 1: Passed
   Step 2: Passed
   Step 3: Passed
   Finalised datetime string = 2020:01:08 20:49:03
   Step 4: Passed
   Step 5: Passed
Reading:  D:\lara\Snapchat\snapchat_downloads_2020_2021_renamed\2020-01-08_205236 - Snapchat.jpg
   Step 1: Passed
   Step 2: Passed
   Step 3: Passed
   Finalised datetime string = 2020:01:08 20:52:36
   Step 4: Passed
   Step 5: Passed
Reading:  D:\lara\Snapchat\snapchat_downloads_2020_2021_renamed\2020-01-08_205257 - Snapchat.jpg
   Step 1: Passed
   Step 2: Passed
   Step 3: Passed
   Finalised datetime string = 2020:01:08 20:52:57
   Step 4: Passed
   Step 5: Passed
Reading:  D:\lara\Snapchat\sna

Reading:  D:\lara\Snapchat\snapchat_downloads_2020_2021_renamed\2020-01-14_173034_1_main - Snapchat.jpg
   Step 1: Passed
   Step 2: Passed
   Step 3: Passed
   Finalised datetime string = 2020:01:14 17:30:34
   Step 4: Passed
   Step 5: Passed
Reading:  D:\lara\Snapchat\snapchat_downloads_2020_2021_renamed\2020-01-14_173129 - Snapchat.jpg
   Step 1: Passed
   Step 2: Passed
   Step 3: Passed
   Finalised datetime string = 2020:01:14 17:31:29
   Step 4: Passed
   Step 5: Passed
Reading:  D:\lara\Snapchat\snapchat_downloads_2020_2021_renamed\2020-01-14_173531 - Snapchat.jpg
   Step 1: Passed
   Step 2: Passed
   Step 3: Passed
   Finalised datetime string = 2020:01:14 17:35:31
   Step 4: Passed
   Step 5: Passed
Reading:  D:\lara\Snapchat\snapchat_downloads_2020_2021_renamed\2020-01-14_191321 - Snapchat.jpg
   Step 1: Passed
   Step 2: Passed
   Step 3: Passed
   Finalised datetime string = 2020:01:14 19:13:21
   Step 4: Passed
   Step 5: Passed
Reading:  D:\lara\Snapchat\snapchat_downl

Reading:  D:\lara\Snapchat\snapchat_downloads_2020_2021_renamed\2020-01-21_180146 - Snapchat.jpg
   Step 1: Passed
   Step 2: Passed
   Step 3: Passed
   Finalised datetime string = 2020:01:21 18:01:46
   Step 4: Passed
   Step 5: Passed
Reading:  D:\lara\Snapchat\snapchat_downloads_2020_2021_renamed\2020-01-21_194232_1_main - Snapchat.jpg
   Step 1: Passed
   Step 2: Passed
   Step 3: Passed
   Finalised datetime string = 2020:01:21 19:42:32
   Step 4: Passed
   Step 5: Passed
Reading:  D:\lara\Snapchat\snapchat_downloads_2020_2021_renamed\2020-01-21_211747_1_main - Snapchat.jpg
   Step 1: Passed
   Step 2: Passed
   Step 3: Passed
   Finalised datetime string = 2020:01:21 21:17:47
   Step 4: Passed
   Step 5: Passed
Reading:  D:\lara\Snapchat\snapchat_downloads_2020_2021_renamed\2020-01-21_225716_1_main - Snapchat.jpg
   Step 1: Passed
   Step 2: Passed
   Step 3: Passed
   Finalised datetime string = 2020:01:21 22:57:16
   Step 4: Passed
   Step 5: Passed
Reading:  D:\lara\Snapchat\

   Step 5: Passed
Reading:  D:\lara\Snapchat\snapchat_downloads_2020_2021_renamed\2020-01-28_145752 - Snapchat.jpg
   Step 1: Passed
   Step 2: Passed
   Step 3: Passed
   Finalised datetime string = 2020:01:28 14:57:52
   Step 4: Passed
   Step 5: Passed
Reading:  D:\lara\Snapchat\snapchat_downloads_2020_2021_renamed\2020-01-28_145812 - Snapchat.jpg
   Step 1: Passed
   Step 2: Passed
   Step 3: Passed
   Finalised datetime string = 2020:01:28 14:58:12
   Step 4: Passed
   Step 5: Passed
Reading:  D:\lara\Snapchat\snapchat_downloads_2020_2021_renamed\2020-01-28_163422_1_main - Snapchat.jpg
   Step 1: Passed
   Step 2: Passed
   Step 3: Passed
   Finalised datetime string = 2020:01:28 16:34:22
   Step 4: Passed
   Step 5: Passed
Reading:  D:\lara\Snapchat\snapchat_downloads_2020_2021_renamed\2020-01-28_232753 - Snapchat.jpg
   Step 1: Passed
   Step 2: Passed
   Step 3: Passed
   Finalised datetime string = 2020:01:28 23:27:53
   Step 4: Passed
   Step 5: Passed
Reading:  D:\lara\Snapc

...

   Step 5: Passed
Reading:  D:\lara\Snapchat\snapchat_downloads_2020_2021_renamed\2021-12-06_140454_1_main - Snapchat.jpg
   Step 1: Passed
   Step 2: Passed
   Step 3: Passed
   Finalised datetime string = 2021:12:06 14:04:54
   Step 4: Passed
   Step 5: Passed
Reading:  D:\lara\Snapchat\snapchat_downloads_2020_2021_renamed\2021-12-06_140507_1_main - Snapchat.jpg
   Step 1: Passed
   Step 2: Passed
   Step 3: Passed
   Finalised datetime string = 2021:12:06 14:05:07
   Step 4: Passed
   Step 5: Passed
Reading:  D:\lara\Snapchat\snapchat_downloads_2020_2021_renamed\2021-12-06_140832_1_main - Snapchat.jpg
   Step 1: Passed
   Step 2: Passed
   Step 3: Passed
   Finalised datetime string = 2021:12:06 14:08:32
   Step 4: Passed
   Step 5: Passed
Reading:  D:\lara\Snapchat\snapchat_downloads_2020_2021_renamed\2021-12-06_140943_1_main - Snapchat.jpg
   Step 1: Passed
   Step 2: Passed
   Step 3: Passed
   Finalised datetime string = 2021:12:06 14:09:43
   Step 4: Passed
   Step 5: Passed
Re

   Step 4: Passed
   Step 5: Passed
Reading:  D:\lara\Snapchat\snapchat_downloads_2020_2021_renamed\2021-12-10_014441 - Snapchat.jpg
   Step 1: Passed
   Step 2: Passed
   Step 3: Passed
   Finalised datetime string = 2021:12:10 01:44:41
   Step 4: Passed
   Step 5: Passed
Reading:  D:\lara\Snapchat\snapchat_downloads_2020_2021_renamed\2021-12-10_014444 - Snapchat.jpg
   Step 1: Passed
   Step 2: Passed
   Step 3: Passed
   Finalised datetime string = 2021:12:10 01:44:44
   Step 4: Passed
   Step 5: Passed
Reading:  D:\lara\Snapchat\snapchat_downloads_2020_2021_renamed\2021-12-10_014444_1_1_main - Snapchat.jpg
   Step 1: Passed
   Step 2: Passed
   Step 3: Passed
   Finalised datetime string = 2021:12:10 01:44:44
   Step 4: Passed
   Step 5: Passed
Reading:  D:\lara\Snapchat\snapchat_downloads_2020_2021_renamed\2021-12-10_014444_1_main - Snapchat.jpg
   Step 1: Passed
   Step 2: Passed
   Step 3: Passed
   Finalised datetime string = 2021:12:10 01:44:44
   Step 4: Passed
   Step 5: Pas

   Step 5: Passed
Reading:  D:\lara\Snapchat\snapchat_downloads_2020_2021_renamed\2021-12-11_185431 - Snapchat.jpg
   Step 1: Passed
   Step 2: Passed
   Step 3: Passed
   Finalised datetime string = 2021:12:11 18:54:31
   Step 4: Passed
   Step 5: Passed
Reading:  D:\lara\Snapchat\snapchat_downloads_2020_2021_renamed\2021-12-11_185441 - Snapchat.jpg
   Step 1: Passed
   Step 2: Passed
   Step 3: Passed
   Finalised datetime string = 2021:12:11 18:54:41
   Step 4: Passed
   Step 5: Passed
Reading:  D:\lara\Snapchat\snapchat_downloads_2020_2021_renamed\2021-12-11_185450 - Snapchat.jpg
   Step 1: Passed
   Step 2: Passed
   Step 3: Passed
   Finalised datetime string = 2021:12:11 18:54:50
   Step 4: Passed
   Step 5: Passed
Reading:  D:\lara\Snapchat\snapchat_downloads_2020_2021_renamed\2021-12-11_185455 - Snapchat.jpg
   Step 1: Passed
   Step 2: Passed
   Step 3: Passed
   Finalised datetime string = 2021:12:11 18:54:55
   Step 4: Passed
   Step 5: Passed
Reading:  D:\lara\Snapchat\sna

Reading:  D:\lara\Snapchat\snapchat_downloads_2020_2021_renamed\2021-12-20_231343_1_main - Snapchat.jpg
   Step 1: Passed
   Step 2: Passed
   Step 3: Passed
   Finalised datetime string = 2021:12:20 23:13:43
   Step 4: Passed
   Step 5: Passed
Reading:  D:\lara\Snapchat\snapchat_downloads_2020_2021_renamed\2021-12-21_002740_1_main - Snapchat.jpg
   Step 1: Passed
   Step 2: Passed
   Step 3: Passed
   Finalised datetime string = 2021:12:21 00:27:40
   Step 4: Passed
   Step 5: Passed
Reading:  D:\lara\Snapchat\snapchat_downloads_2020_2021_renamed\2021-12-21_144810_1_main - Snapchat.jpg
   Step 1: Passed
   Step 2: Passed
   Step 3: Passed
   Finalised datetime string = 2021:12:21 14:48:10
   Step 4: Passed
   Step 5: Passed
Reading:  D:\lara\Snapchat\snapchat_downloads_2020_2021_renamed\2021-12-21_144838_1_main - Snapchat.jpg
   Step 1: Passed
   Step 2: Passed
   Step 3: Passed
   Finalised datetime string = 2021:12:21 14:48:38
   Step 4: Passed
   Step 5: Passed
Reading:  D:\lara\Sn

Reading:  D:\lara\Snapchat\snapchat_downloads_2020_2021_renamed\2021-12-28_230331 - Snapchat.jpg
   Step 1: Passed
   Step 2: Passed
   Step 3: Passed
   Finalised datetime string = 2021:12:28 23:03:31
   Step 4: Passed
   Step 5: Passed
Reading:  D:\lara\Snapchat\snapchat_downloads_2020_2021_renamed\2021-12-29_020104 - Snapchat.jpg
   Step 1: Passed
   Step 2: Passed
   Step 3: Passed
   Finalised datetime string = 2021:12:29 02:01:04
   Step 4: Passed
   Step 5: Passed
Reading:  D:\lara\Snapchat\snapchat_downloads_2020_2021_renamed\2021-12-29_205511 - Snapchat.jpg
   Step 1: Passed
   Step 2: Passed
   Step 3: Passed
   Finalised datetime string = 2021:12:29 20:55:11
   Step 4: Passed
   Step 5: Passed
Reading:  D:\lara\Snapchat\snapchat_downloads_2020_2021_renamed\2021-12-30_004840_1_main - Snapchat.jpg
   Step 1: Passed
   Step 2: Passed
   Step 3: Passed
   Finalised datetime string = 2021:12:30 00:48:40
   Step 4: Passed
   Step 5: Passed
Reading:  D:\lara\Snapchat\snapchat_downl